# Distillation through augmented noise — ViT-8 → ViT-4 (CIFAR-10)

The ViT counterpart of the AlexNet → AlexNet-Half notebook. The data-free distillation
pipeline is unchanged: synthetic noise images (smooth gradient / Perlin / uniform / Gabor /
checkerboard) go through the same 17-op augmentation stack, the frozen teacher labels the
augmented view on the fly, and the student is trained against those labels with the same
temperature-scaled KL objective, batch-size ramp and phase-aligned cosine LR schedule. No
CIFAR-10 training image ever reaches the student.

## What "ViT-8 → ViT-4" means here

The student halves the teacher's **depth**: 8 transformer blocks → 4, with every other
dimension (patch size, embedding width, head count, MLP ratio) held fixed. That mirrors how
AlexNet-Half halves AlexNet, and depth is the usual compression axis for ViT distillation.
Halving the *patch* size instead (8 → 4) would quadruple the token count and make the
student larger than the teacher, which is backwards for distillation — so it is read as
depth here. `TEACHER_DEPTH` / `STUDENT_DEPTH` are constants in the model cell if you meant
a different split.

| | patch | dim | heads | MLP ratio | blocks | params |
|---|---|---|---|---|---|---|
| ViT-8 (teacher) | 4 | 256 | 4 | 2.0 | 8 | 4,249,354 |
| ViT-4 (student) | 4 | 256 | 4 | 2.0 | 4 | 2,140,938 |

A patch size of 4 on a 32×32 image gives 8×8 = 64 patches, plus a CLS token — the standard
CIFAR ViT setup. Torchvision's `vit_b_16` is an ImageNet 224×224 design and is not usable
at this resolution, so the architecture is defined here.

## Teacher training

The teacher is trained on CIFAR-10 from scratch, which is where this notebook departs from
the AlexNet one. A from-scratch ViT does not train under the AlexNet recipe (SGD at lr 0.01
with crop+flip): it needs **AdamW, a warmup then cosine schedule, weight decay 0.05,
label smoothing, gradient clipping**, and stronger augmentation. All of that is confined to
the teacher — the distillation half of the pipeline is byte-identical to the AlexNet
notebook, so the two experiments stay comparable.

Set `TEACHER_RANDAUG = False` for exactly the AlexNet notebook's crop+flip teacher
augmentation; expect a few points less teacher accuracy.

## 1. Setup

In [ ]:
import math
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True          # fixed 32x32 input size -> free speedup

# Worker/precision settings picked from the hardware actually available. None of
# this touches the training math.
_cpu_count = os.cpu_count() or 2
NUM_WORKERS = max(0, min(16, _cpu_count - 1))
PREFETCH_FACTOR = 2 if NUM_WORKERS <= 2 else (4 if NUM_WORKERS <= 8 else 6)
BF16_OK = device.type == "cuda" and torch.cuda.is_bf16_supported()
torch.set_num_threads(max(1, min(_cpu_count, 32)))
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ---- what hardware is this actually running on? -----------------------------
# The student batch ramp climbs to 2048; on a smaller card that will OOM, so the
# ramp is capped by available VRAM. Its shape is unchanged -- it just stops
# climbing past what fits. Set MAX_BATCH by hand to override.
if device.type == "cuda":
    _props = torch.cuda.get_device_properties(0)
    VRAM_GB = _props.total_memory / 1e9
    MAX_BATCH = 2048 if VRAM_GB >= 16 else (1024 if VRAM_GB >= 10 else 512)
    print(f"GPU: {_props.name}  {VRAM_GB:.1f} GB  "
          f"compute capability {_props.major}.{_props.minor}")
    print(f"precision: {'bf16' if BF16_OK else 'fp16 + GradScaler'}")
else:
    VRAM_GB, MAX_BATCH = 0.0, 256
    print("GPU: none found -- running on CPU. Teacher training will be impractically "
          "slow; use a GPU runtime for the 200-epoch run.")

print(f"Device: {device}")
print(f"NUM_WORKERS={NUM_WORKERS} PREFETCH_FACTOR={PREFETCH_FACTOR} "
      f"BF16_OK={BF16_OK} MAX_BATCH={MAX_BATCH}")


# ---- shared fast-path helpers (used by both training loops) ------------------
def _prepare_for_fast(model, device):
    # channels_last is a pure memory-layout change; it only touches the conv
    # patch embedding here, and conv math is identical either way.
    return model.to(memory_format=torch.channels_last) if device.type == "cuda" else model


def _fast_input(x, device):
    return x.contiguous(memory_format=torch.channels_last) if device.type == "cuda" else x


def _train_autocast(device):
    # bf16 where supported (no GradScaler needed -- bf16 has fp32's exponent range
    # so it doesn't underflow like fp16); fp16+scaler on older GPUs; fp32 on CPU.
    if device.type == "cuda" and BF16_OK:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16), False
    if device.type == "cuda":
        return torch.amp.autocast("cuda"), True
    from contextlib import nullcontext
    return nullcontext(), False

## 2. ViT-8 / ViT-4

A CIFAR-sized vision transformer: 4×4 patches over a 32×32 image (64 tokens), a CLS token,
learned position embeddings, and pre-norm blocks. Attention goes through
`F.scaled_dot_product_attention`, so it picks the fused kernel when one is available.

In [ ]:
class Attention(nn.Module):
    """Multi-head self-attention. Pre-norm is applied by the enclosing Block."""

    def __init__(self, dim, heads, drop=0.0):
        super().__init__()
        assert dim % heads == 0, "dim must be divisible by heads"
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = drop
        self.proj_drop = nn.Dropout(drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.attn_drop if self.training else 0.0)
        return self.proj_drop(self.proj(x.transpose(1, 2).reshape(B, N, C)))


class Block(nn.Module):
    """Pre-norm transformer block: x + attn(norm(x)), then x + mlp(norm(x))."""

    def __init__(self, dim, heads, mlp_ratio=2.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads, drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class ViT(nn.Module):
    """Vision transformer sized for 32x32 CIFAR images.

    Only `depth` differs between teacher and student -- patch size, width, heads
    and MLP ratio are held fixed, so the student is the teacher with half the
    blocks.
    """

    def __init__(self, depth, num_classes=10, img_size=32, patch=4, dim=256,
                 heads=4, mlp_ratio=2.0, drop=0.1):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)
        n_patches = (img_size // patch) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        self.pos_drop = nn.Dropout(drop)

        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio, drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)        # (B, N, dim)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = self.pos_drop(torch.cat((cls, x), dim=1) + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])                      # CLS token


TEACHER_DEPTH, STUDENT_DEPTH = 8, 4


def ViT8(num_classes=10, **kw):
    """Teacher: 8 transformer blocks."""
    return ViT(TEACHER_DEPTH, num_classes, **kw)


def ViT4(num_classes=10, **kw):
    """Student: the same network with half the blocks."""
    return ViT(STUDENT_DEPTH, num_classes, **kw)


for _name, _net in [("ViT-8 (teacher)", ViT8()), ("ViT-4 (student)", ViT4())]:
    print(f"{_name:20s} {sum(p.numel() for p in _net.parameters()):>9,d} params")

## 3. CIFAR-10

`DATA_ROOT` must be the directory that *contains* `cifar-10-batches-py`, not that folder
itself — torchvision resolves the data as `<root>/cifar-10-batches-py/...`.
`TEACHER_RANDAUG` adds RandAugment + random erasing to the **teacher's** training
transform only; the student never sees CIFAR images, so it does not affect distillation.

In [ ]:
NUM_CLASSES = 10
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

DATA_ROOT = '/home/vu-lab03-pc43/Downloads/data'
DOWNLOAD = False

if not DOWNLOAD and not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py')):
    raise FileNotFoundError(
        f"{DATA_ROOT} does not contain 'cifar-10-batches-py'. DATA_ROOT must be the "
        f"PARENT of that folder, not the folder itself. Set DOWNLOAD = True to fetch it."
    )

TEACHER_RANDAUG = True     # False -> the AlexNet notebook's plain crop+flip

_base_aug = [T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()]
_extra_aug = [T.RandAugment(num_ops=2, magnitude=9)] if TEACHER_RANDAUG else []
_post_aug = [T.RandomErasing(p=0.25)] if TEACHER_RANDAUG else []

transform_train = T.Compose(_base_aug + _extra_aug + [T.ToTensor()] + _post_aug)
transform_test = T.Compose([T.ToTensor()])

train_set = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=DOWNLOAD, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=DOWNLOAD, transform=transform_test)
# 128 pairs with the teacher's AdamW lr of 1e-3; workers matter because
# RandAugment runs on CPU and would otherwise starve the GPU.
TEACHER_BATCH = 128

_loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=PREFETCH_FACTOR)

train_loader = DataLoader(train_set, batch_size=TEACHER_BATCH, shuffle=True, **_loader_kw)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, **_loader_kw)
print(f"CIFAR-10: {len(train_set)} train / {len(test_set)} test")


@torch.no_grad()
def evaluate(model, device, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (torch.argmax(model(x), dim=1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


class normalization(nn.Module):
    """Wraps a backbone so it takes [0, 1] images and normalizes internally --
    the noise pipeline works in [0, 1] throughout."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        mean = torch.tensor(CIFAR_MEAN, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
        std = torch.tensor(CIFAR_STD, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
        return self.backbone((x - mean) / std)

## 4. Teacher — trained from scratch

`train_teacher` uses the recipe a from-scratch ViT needs: AdamW, linear warmup into a
cosine decay, weight decay 0.05, label smoothing 0.1, gradient clipping at 1.0, and the
same AMP path as the student loop. It saves the best checkpoint as it goes, so an
interrupted run can be resumed by re-running the cell.

`load_teacher_checkpoint` absorbs the usual checkpoint-shape differences: a container dict
(`{'model_state': ..., 'val_acc': ..., 'epoch': ...}`) instead of a bare `state_dict`, a
`module.` prefix from `DataParallel`, and the `backbone.` prefix that `normalization(...)`
adds but a checkpoint saved from the bare backbone lacks.

In [ ]:
_CONTAINER_KEYS = ("model_state", "state_dict", "model", "net", "weights", "teacher")
_META_KEYS = ("val_acc", "test_acc", "train_acc", "run_acc", "acc", "best_acc",
              "epoch", "epochs")


def _strip(prefix):
    return lambda k: k[len(prefix):] if k.startswith(prefix) else k


_KEY_FIXES = [
    ("as-is", lambda k: k),
    ("add 'backbone.'", lambda k: f"backbone.{k}"),
    ("strip 'backbone.'", _strip("backbone.")),
]


def _candidate_state_dicts(obj):
    """Yield every plausible state_dict inside a checkpoint object."""
    if not isinstance(obj, dict):
        return
    if any(torch.is_tensor(v) for v in obj.values()):
        yield obj
    for k in _CONTAINER_KEYS:
        v = obj.get(k)
        if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
            yield v


def load_teacher_checkpoint(model, path, device):
    """Load `path` into `model`, reconciling container dicts and key prefixes.
    Raises with the keys it saw when nothing matches, rather than a wall of
    missing-key output."""
    try:
        obj = torch.load(path, map_location=device, weights_only=False)
    except TypeError:                      # torch too old for weights_only
        obj = torch.load(path, map_location=device)

    meta = {k: obj[k] for k in _META_KEYS if isinstance(obj, dict) and k in obj}
    want = model.state_dict()
    best = (-1, None, None)

    for sd in _candidate_state_dicts(obj):
        sd = {_strip("module.")(k): v for k, v in sd.items()}
        for name, fix in _KEY_FIXES:
            cand = {fix(k): v for k, v in sd.items()}
            overlap = len(set(want) & set(cand))
            if overlap > best[0]:
                best = (overlap, name, cand)
            if set(cand) == set(want):
                bad = {k: (tuple(v.shape), tuple(want[k].shape))
                       for k, v in cand.items() if v.shape != want[k].shape}
                if bad:
                    k, (got, exp) = next(iter(bad.items()))
                    raise RuntimeError(
                        f"{path} matches the architecture but not its shapes "
                        f"({len(bad)} tensor(s) differ, e.g. {k}: checkpoint {got} vs "
                        f"model {exp}). A classifier mismatch means the checkpoint was "
                        f"trained on a different dataset."
                    )
                model.load_state_dict(cand)
                print(f"  loaded {path} (keys matched {name})"
                      + (f"\n  checkpoint metadata: {meta}" if meta else ""))
                return model

    overlap, name, cand = best
    raise RuntimeError(
        f"Could not match {path} to this model.\n"
        f"  best attempt ({name}) matched {overlap}/{len(want)} keys\n"
        f"  checkpoint keys (first 5): {sorted(cand)[:5] if cand else 'none found'}\n"
        f"  model keys (first 5):      {sorted(want)[:5]}\n"
        f"  Unrelated names mean the checkpoint is a different architecture."
    )


def make_train_eval_loader(train_set, transform_test, batch_size=256, subset=10000,
                           seed=0, **loader_kwargs):
    """A loader over the TRAINING set with the TEST transform applied.

    Needed because `train_loader` applies random crops/flips (and RandAugment),
    so accuracy measured on it conflates fit with augmentation strength. Scoring
    a clean view is what makes the train/test gap readable as overfitting.

    `subset` caps the number of images so the extra pass stays cheap; 0 uses all
    of them.
    """
    import copy

    clean = copy.copy(train_set)          # shallow: shares the underlying data
    clean.transform = transform_test

    if 0 < subset < len(clean):
        generator = torch.Generator().manual_seed(seed)
        picks = torch.randperm(len(clean), generator=generator)[:subset].tolist()
        clean = torch.utils.data.Subset(clean, picks)

    return torch.utils.data.DataLoader(clean, batch_size=batch_size, shuffle=False,
                                       **loader_kwargs)


def train_teacher(teacher, train_loader, test_loader, device, epochs=200, lr=1e-3,
                  weight_decay=0.05, warmup=5, label_smoothing=0.1, clip_grad=1.0,
                  ckpt_path=None, train_eval_loader=None, eval_train_every=1,
                  log_every=1, history=None):
    """Supervised training from scratch, saving the best model on the fly.

    AdamW + warmup + cosine rather than the AlexNet notebook's SGD: a ViT trained
    from scratch on CIFAR-10 does not converge under the SGD recipe.

    Reports THREE accuracies, because two different things get called "train
    accuracy" and they answer different questions:

      run    accuracy over the training pass itself. Free -- the logits are
             already computed -- but measured on AUGMENTED inputs while the
             weights are still moving, so it reads low early and mostly tracks
             how hard the augmentation is.
      train  a separate pass over `train_eval_loader`, which should apply the
             TEST transform to the training set (see make_train_eval_loader).
             This is the number to compare against test accuracy: the gap
             between them is overfitting. Skipped if no loader is given.
      test   accuracy on the test set; this is what selects the checkpoint.

    Pass a list as `history` to collect a per-epoch record for plotting.
    """
    teacher = _prepare_for_fast(teacher.to(device), device)
    opt = torch.optim.AdamW(teacher.parameters(), lr=lr, weight_decay=weight_decay)

    def lr_lambda(epoch):                       # linear warmup, then cosine to ~0
        if warmup and epoch < warmup:
            return (epoch + 1) / warmup
        t = (epoch - warmup) / max(epochs - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    autocast_ctx, needs_scaler = _train_autocast(device)
    scaler = torch.amp.GradScaler("cuda", enabled=needs_scaler)

    if ckpt_path:
        os.makedirs(os.path.dirname(os.path.abspath(ckpt_path)), exist_ok=True)
    if train_eval_loader is None:
        print("[teacher] no train_eval_loader given -- reporting the running "
              "(augmented) train accuracy only; see make_train_eval_loader")

    best_acc, best_train = 0.0, None
    for epoch in range(epochs):
        teacher.train()
        epoch_loss, correct, seen = 0.0, 0, 0

        for x, y in train_loader:
            x = _fast_input(x.to(device, non_blocking=True), device)
            y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast_ctx:
                logits = teacher(x)
                loss = ce(logits, y)
            if needs_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(teacher.parameters(), clip_grad)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(teacher.parameters(), clip_grad)
                opt.step()

            epoch_loss += loss.item()
            correct += (logits.detach().argmax(dim=1) == y).sum().item()
            seen += y.size(0)
        sched.step()

        run_acc = 100.0 * correct / max(seen, 1)
        test_acc = evaluate(teacher, device, test_loader)
        measure_train = (train_eval_loader is not None and eval_train_every
                         and (epoch + 1) % eval_train_every == 0)
        train_acc = evaluate(teacher, device, train_eval_loader) if measure_train else None

        if test_acc > best_acc:
            best_acc, best_train = test_acc, train_acc
            if ckpt_path:
                torch.save({"model_state": teacher.state_dict(),
                            "test_acc": test_acc, "val_acc": test_acc,
                            "train_acc": train_acc, "run_acc": run_acc,
                            "epoch": epoch + 1}, ckpt_path)

        if history is not None:
            history.append({"epoch": epoch + 1, "run_acc": run_acc,
                            "train_acc": train_acc, "test_acc": test_acc,
                            "loss": epoch_loss / len(train_loader),
                            "lr": sched.get_last_lr()[0]})

        if log_every and ((epoch + 1) % log_every == 0 or epoch == 0
                          or epoch == epochs - 1):
            line = f"[teacher] epoch {epoch + 1}/{epochs}  run={run_acc:.2f}%  "
            if train_acc is not None:
                line += f"train={train_acc:.2f}%  "
            line += (f"test={test_acc:.2f}%  best={best_acc:.2f}%  "
                     f"loss={epoch_loss / len(train_loader):.4f}  "
                     f"lr={sched.get_last_lr()[0]:.2e}")
            if train_acc is not None:
                line += f"  gap={train_acc - test_acc:+.2f}"
            print(line)

    summary = f"[teacher] done. best test={best_acc:.2f}%"
    if best_train is not None:
        summary += (f"  (train={best_train:.2f}% at that epoch, "
                    f"gap={best_train - best_acc:+.2f})")
    if ckpt_path:
        summary += f"  -> {ckpt_path}"
    print(summary)
    return teacher, best_acc

In [ ]:
TEACHER_CKPT = '/home/vu-lab03-pc43/Downloads/ViT8_cifar10_teacher_best.pt'
TEACHER_EPOCHS = 200

teacher = normalization(ViT8(NUM_CLASSES)).to(device)

if os.path.exists(TEACHER_CKPT):
    print(f"Found a teacher checkpoint -> {TEACHER_CKPT}")
    load_teacher_checkpoint(teacher, TEACHER_CKPT, device)
    teacher.to(device)
else:
    print(f"No checkpoint at {TEACHER_CKPT}\n"
          f"Training ViT-8 from scratch for {TEACHER_EPOCHS} epochs "
          f"(best epoch is saved as it goes, so an interrupted run resumes by "
          f"re-running this cell).")
    train_eval_loader = make_train_eval_loader(
        train_set, transform_test, batch_size=256, subset=10000, **_loader_kw)
    teacher, _ = train_teacher(teacher, train_loader, test_loader, device,
                               epochs=TEACHER_EPOCHS, ckpt_path=TEACHER_CKPT,
                               train_eval_loader=train_eval_loader)
    # reload the best epoch rather than keeping the last one
    load_teacher_checkpoint(teacher, TEACHER_CKPT, device)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

print(f"teacher accuracy: {evaluate(teacher, device, test_loader):.2f}%")

## 5. Noise

In [ ]:
def noise_smooth_gradient(n, size=32):
    """Smooth linear gradient at a random angle between two random colours."""
    yy, xx = torch.meshgrid(torch.linspace(0, 1, size), torch.linspace(0, 1, size), indexing="ij")
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        angle = random.uniform(0, 2 * math.pi)
        grad = xx * math.cos(angle) + yy * math.sin(angle)
        imgs[i] = _colourise(grad)
    return imgs.clamp(0.0, 1.0)


def noise_perlin(n, size=32):
    """Perlin noise field per image, at a coarse/medium/fine cell grid."""
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        imgs[i] = _colourise(_perlin_grid(size, random.choice([2, 4, 8])))
    return imgs.clamp(0.0, 1.0)


def noise_uniform(n, size=32):
    """Plain i.i.d. uniform noise -- no spatial smoothness, unlike the others."""
    return torch.rand(n, 3, size, size).clamp(0.0, 1.0)


def noise_gabor(n, size=32):
    """Gaussian-windowed sinusoidal grating, random orientation/frequency/phase."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing="ij")
    for i in range(n):
        theta = random.uniform(0, math.pi)
        freq = random.uniform(2.0, 8.0)
        phase = random.uniform(0, 2 * math.pi)
        sigma = random.uniform(0.3, 0.8)
        x_theta = xx * math.cos(theta) + yy * math.sin(theta)
        y_theta = -xx * math.sin(theta) + yy * math.cos(theta)
        gaussian = torch.exp(-(x_theta ** 2 + y_theta ** 2) / (2 * sigma ** 2))
        imgs[i] = _colourise(gaussian * torch.cos(2 * math.pi * freq * x_theta + phase))
    return imgs.clamp(0.0, 1.0)


def noise_checkerboard(n, size=32):
    """Checkerboard with random cell size and random phase offset per image."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    for i in range(n):
        block = random.choice([2, 4, 8, 16])
        ox, oy = random.randint(0, block - 1), random.randint(0, block - 1)
        imgs[i] = _colourise((((xx + ox) // block) + ((yy + oy) // block)) % 2, norm=False)
    return imgs.clamp(0.0, 1.0)


def _colourise(grad, norm=True):
    """Map a scalar field to a 3-channel image interpolating two random colours."""
    grad = grad.float()
    if norm:
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
    color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
    return grad.unsqueeze(0) * color_a + (1 - grad.unsqueeze(0)) * color_b


def _perlin_grid(size, res):
    """Single 2D Perlin field, values roughly in [-1, 1]. `size` must divide by `res`."""
    assert size % res == 0, "size must be divisible by res"
    d = size // res
    lin = torch.arange(0, res, 1.0 / d)
    gy, gx = torch.meshgrid(lin, lin, indexing="ij")
    grid = torch.stack((gy % 1, gx % 1), dim=-1)

    angles = 2 * math.pi * torch.rand(res + 1, res + 1)
    grads = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)
    tile = lambda g: g.repeat_interleave(d, 0).repeat_interleave(d, 1)

    g00, g10 = tile(grads[:-1, :-1]), tile(grads[1:, :-1])
    g01, g11 = tile(grads[:-1, 1:]), tile(grads[1:, 1:])
    n00 = (torch.stack((grid[..., 0],     grid[..., 1]),     -1) * g00).sum(-1)
    n10 = (torch.stack((grid[..., 0] - 1, grid[..., 1]),     -1) * g10).sum(-1)
    n01 = (torch.stack((grid[..., 0],     grid[..., 1] - 1), -1) * g01).sum(-1)
    n11 = (torch.stack((grid[..., 0] - 1, grid[..., 1] - 1), -1) * g11).sum(-1)

    t = 6 * grid ** 5 - 15 * grid ** 4 + 10 * grid ** 3          # fade curve
    return torch.lerp(torch.lerp(n00, n10, t[..., 0]),
                      torch.lerp(n01, n11, t[..., 0]), t[..., 1])


NOISE_INITIALIZERS = {
    "noise_smooth": noise_smooth_gradient,
    "noise_perlin": noise_perlin,
    "noise_uniform": noise_uniform,
    "noise_gabor": noise_gabor,
    "noise_checkerboard": noise_checkerboard,
}


def noise_data(p, per_call=100):
    """p * per_call synthetic images, noise family drawn per call."""
    fns = list(NOISE_INITIALIZERS.values())
    return torch.cat([random.choice(fns)(per_call) for _ in range(p)], dim=0)

## 6. Augmentation

Two independent switches, applied in order: a geometric base (`RandomCrop(32, pad=4,
reflect)` + `RandomHorizontalFlip`), then `n_random_ops` sampled without replacement from
the 17-op pool. Both happen inside `Dataset.__getitem__`, so the teacher is queried on the
augmented view.

In [ ]:
base_geo_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

_perspective_tf = T.RandomPerspective(distortion_scale=0.35, p=1.0)
_zoom_crop_tf = T.RandomResizedCrop(32, scale=(0.65, 1.0), ratio=(0.85, 1.15))
_color_jitter_tf = T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.45, hue=0.12)
_cutout_tf = T.RandomErasing(p=1.0, scale=(0.02, 0.25), ratio=(0.3, 3.3), value=0.0)


def op_rotate(img):
    return T.functional.rotate(img, random.uniform(-20, 20))


def op_affine(img):
    return T.functional.affine(
        img,
        angle=random.uniform(-12, 12),
        translate=(random.randint(-2, 2), random.randint(-2, 2)),
        scale=random.uniform(0.82, 1.18),
        shear=random.uniform(-12, 12),
    )


def op_gaussian_blur(img):
    return T.functional.gaussian_blur(img, kernel_size=random.choice([3, 5]),
                                      sigma=random.uniform(0.1, 2.2))


def op_equalize(img):
    return T.functional.equalize((img.clamp(0.0, 1.0) * 255).to(torch.uint8)).float() / 255.0


def op_posterize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.posterize(img_u8, random.choice([3, 4, 5, 6])).float() / 255.0


def op_gaussian_noise(img):
    return (img + torch.randn_like(img) * random.uniform(0.01, 0.08)).clamp(0.0, 1.0)


def op_salt_pepper(img):
    prob = random.uniform(0.01, 0.06)
    mask = torch.rand(1, img.shape[1], img.shape[2], device=img.device)
    out = img.clone()
    out[(mask < prob / 2).expand_as(img)] = 1.0
    out[(mask > 1 - prob / 2).expand_as(img)] = 0.0
    return out


def op_random_color_erase(img):
    out = img.clone()
    eh, ew = random.randint(4, 12), random.randint(4, 12)
    y0 = random.randint(0, img.shape[1] - eh)
    x0 = random.randint(0, img.shape[2] - ew)
    out[:, y0:y0 + eh, x0:x0 + ew] = torch.rand(3, 1, 1, device=img.device)
    return out


AUG_OPS = {
    "rotate": op_rotate,
    "affine": op_affine,
    "perspective": _perspective_tf,
    "zoom_crop": _zoom_crop_tf,
    "color_jitter": _color_jitter_tf,
    "grayscale": lambda img: T.functional.rgb_to_grayscale(img, num_output_channels=3),
    "gaussian_blur": op_gaussian_blur,
    "sharpness": lambda img: T.functional.adjust_sharpness(img, random.uniform(0.0, 3.5)),
    "autocontrast": lambda img: T.functional.autocontrast(img.clamp(0.0, 1.0)),
    "equalize": op_equalize,
    "posterize": op_posterize,
    "solarize": lambda img: T.functional.solarize(img.clamp(0.0, 1.0), random.uniform(0.3, 0.9)),
    "invert": lambda img: T.functional.invert(img.clamp(0.0, 1.0)),
    "gaussian_noise": op_gaussian_noise,
    "salt_pepper": op_salt_pepper,
    "cutout": lambda img: _cutout_tf(img.unsqueeze(0)).squeeze(0),
    "random_color_erase": op_random_color_erase,
}
AUG_OP_NAMES = list(AUG_OPS)
assert len(AUG_OP_NAMES) == 17, f"expected a 17-op pool, found {len(AUG_OP_NAMES)}"


def diverse_augment(img, use_geo=True, n_random_ops=4):
    img = img.clamp(0.0, 1.0)
    if use_geo:
        img = base_geo_transform(img)
    for name in random.sample(AUG_OP_NAMES, k=min(int(n_random_ops), len(AUG_OP_NAMES))):
        img = AUG_OPS[name](img).clamp(0.0, 1.0)
    return img


class SyntheticDataset(Dataset):
    """Yields the augmented noise image; the teacher labels it on the fly."""

    def __init__(self, imgs, use_geo=True, n_random_ops=4):
        self.imgs = imgs
        self.use_geo = use_geo
        self.n_random_ops = n_random_ops

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, idx):
        return diverse_augment(self.imgs[idx], self.use_geo, self.n_random_ops)

## 7. Distillation

Identical to the AlexNet notebook, with one addition: `optimizer=` selects AdamW or SGD.
AdamW is the default because the student is a transformer; pass `optimizer="sgd"` for the
exact AlexNet-notebook optimizer.

In [ ]:
def klpga(x, student, t_logits, T):
    """Temperature-scaled KL between student and (frozen) teacher, scaled by T^2."""
    log_prob = F.log_softmax(student(x) / T, dim=-1)
    t_prob = F.softmax(t_logits.detach() / T, dim=-1)
    return F.kl_div(log_prob, t_prob, reduction="batchmean") * T ** 2


_BATCH_RAMP = [(25, 16), (50, 64), (75, 128), (100, 256), (125, 512), (150, 1024)]
_loader_cache = {}


def batch_size_for(epoch):
    """The ramp, capped at what this GPU's VRAM can hold (see the setup cell)."""
    for limit, bs in _BATCH_RAMP:
        if epoch < limit:
            return min(bs, MAX_BATCH)
    return min(2048, MAX_BATCH)


def data_to_loader(data, epoch):
    batch_size = batch_size_for(epoch)
    if batch_size not in _loader_cache:
        kwargs = dict(batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
                      pin_memory=True,
                      drop_last=True)   # constant batch shape -> cudnn.benchmark stays hot
        if NUM_WORKERS > 0:
            kwargs["persistent_workers"] = True
            kwargs["prefetch_factor"] = PREFETCH_FACTOR
        _loader_cache[batch_size] = DataLoader(data, **kwargs)
    return _loader_cache[batch_size]


def batch_aligned_lr(epoch):
    """Cosine decay within each phase, resetting at every batch-size change so the
    schedule lines up with the ramp. Returns a LambdaLR multiplier on lr=0.01."""
    start = 0.001 if 25 <= epoch < 50 else 0.01
    end = 0.00001
    phase_start = min(epoch // 25, 6) * 25
    phase_len = 50 if epoch >= 150 else 25

    t = epoch - phase_start
    cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
    return (end + (start - end) * cos_factor) / 0.01


def _fused_sgd_kwargs(device):
    # Fused CUDA kernel for the optimizer step; same math, same update rule.
    import inspect
    if device.type == "cuda" and "fused" in inspect.signature(torch.optim.SGD.__init__).parameters:
        return {"fused": True}
    return {}


def make_optimizer(params, kind, lr, device):
    """AdamW for the transformer student; SGD reproduces the AlexNet notebook."""
    if kind == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
    if kind == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4,
                               **_fused_sgd_kwargs(device))
    raise ValueError(f"unknown optimizer {kind!r}")


def train_student(teacher, student, dataset, test_loader, device, T,
                  student_epochs=100, lr=0.001, optimizer="adamw", ckpt_path=None):
    """Distil `teacher` into `student` on the synthetic set.

    With ckpt_path set, the best-scoring epoch is written as it goes -- a long run
    that drifts or is interrupted still leaves a usable student on disk.
    """
    student = _prepare_for_fast(student, device)
    teacher = _prepare_for_fast(teacher, device)
    teacher.eval()      # dropout is active in train mode -- keep the labeller deterministic

    opt = make_optimizer(student.parameters(), optimizer, lr, device)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=batch_aligned_lr)
    autocast_ctx, needs_scaler = _train_autocast(device)
    scaler = torch.amp.GradScaler("cuda", enabled=needs_scaler)
    if ckpt_path:
        os.makedirs(os.path.dirname(os.path.abspath(ckpt_path)), exist_ok=True)
    best_acc = 0.0

    for epoch in range(student_epochs):
        student.train()
        loader = data_to_loader(dataset, epoch)
        epoch_loss = 0.0
        for x in loader:
            x = _fast_input(x.to(device, non_blocking=True), device)
            with torch.no_grad(), autocast_ctx:   # teacher already frozen; no_grad also
                z = teacher(x)                    # stops autograd tracking the input side
            opt.zero_grad(set_to_none=True)
            with autocast_ctx:
                loss = klpga(x, student, z, T)
            if needs_scaler:
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()
            epoch_loss += loss.item()
        sched.step()

        acc = evaluate(student, device, test_loader)
        if acc > best_acc:
            best_acc = acc
            if ckpt_path:
                torch.save({"model_state": student.state_dict(), "val_acc": acc,
                            "epoch": epoch + 1, "temperature": T,
                            "arch": f"ViT-{STUDENT_DEPTH}"}, ckpt_path)
        print(f"epoch {epoch + 1}/{student_epochs}  acc={acc:.2f}%  "
              f"best={best_acc:.2f}%  loss={epoch_loss / len(loader):.4f}  "
              f"bs={batch_size_for(epoch)}  lr={sched.get_last_lr()[0]:.2e}")

    print(f"[student] done. best_acc={best_acc:.2f}%"
          + (f" -> {ckpt_path}" if ckpt_path else ""))
    return student

## 8. Run

The diagnostic below prints how peaked the teacher's targets are at the chosen
`TEMPERATURE`. With 10 classes, a large temperature can flatten the softmax so far that
there is almost no signal left to distil — if `mean max prob @ T` sits near 0.10 (chance),
lower it.

The final cell distils and saves the student twice: `train_student` writes the
best-scoring epoch to `STUDENT_CKPT` as it goes, and the final-epoch weights go to
`STUDENT_CKPT_LAST` afterwards. Both use the same container format as the teacher
checkpoint, so `load_teacher_checkpoint` reads either one back — the last cell reloads the
best checkpoint into a fresh ViT-4 to prove the file is usable.

In [ ]:
student = normalization(ViT4(NUM_CLASSES)).to(device)

data = noise_data(1500)
print(f"synthetic set: {tuple(data.shape)}")

dataset = SyntheticDataset(data, use_geo=True, n_random_ops=8)

In [ ]:
TEMPERATURE = 20        # not `T` -- that name is torchvision.transforms in this notebook

with torch.no_grad():
    probe = torch.stack([dataset[i] for i in range(256)]).to(device)
    logits = teacher(probe)
    flat = F.softmax(logits / TEMPERATURE, dim=-1).max(-1).values.mean().item()
    sharp = F.softmax(logits, dim=-1).max(-1).values.mean().item()

print(f"teacher on noise -- mean max prob @ T={TEMPERATURE}: {flat:.4f}   @ T=1: {sharp:.4f}")
print(f"chance is {1 / NUM_CLASSES:.4f}; if the first number is close to it, lower TEMPERATURE")

In [ ]:
STUDENT_CKPT = '/home/vu-lab03-pc43/Downloads/ViT4_cifar10_student_best.pt'
STUDENT_CKPT_LAST = '/home/vu-lab03-pc43/Downloads/ViT4_cifar10_student_last.pt'
STUDENT_EPOCHS = 200

student = train_student(teacher, student, dataset, test_loader, device, T=TEMPERATURE,
                        student_epochs=STUDENT_EPOCHS, lr=1e-3, optimizer="adamw",
                        ckpt_path=STUDENT_CKPT)

# The best epoch is already on disk; also keep the final-epoch weights.
final_acc = evaluate(student, device, test_loader)
torch.save({"model_state": student.state_dict(), "val_acc": final_acc,
            "epoch": STUDENT_EPOCHS, "temperature": TEMPERATURE,
            "arch": f"ViT-{STUDENT_DEPTH} (dim=256, heads=4, patch=4)"},
           STUDENT_CKPT_LAST)

print(f"\nfinal-epoch student: {final_acc:.2f}%  -> {STUDENT_CKPT_LAST}")
print(f"best-epoch student saved -> {STUDENT_CKPT}")

# Confirm the saved file actually reloads into a fresh ViT-4.
reloaded = normalization(ViT4(NUM_CLASSES)).to(device)
load_teacher_checkpoint(reloaded, STUDENT_CKPT, device)
print(f"reloaded best checkpoint: {evaluate(reloaded, device, test_loader):.2f}%")